# 00. Data Preprocessing (ETL)

This notebook handles the "Bronze to Silver" data transformation.
**Objective:** Create a single, clean, wide-format dataset that all state-specific analyses will rely on.

**Steps:**
1. Load Raw Data.
2. Pivot to Wide Format (DateTime Index x Region Columns).
3. **Quality Check:** Ensure the time index is complete (no missing intervals) and handle NaNs.
4. Save 'Silver' dataset for downstream modeling.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add parent directory to path to import src
sys.path.append(os.path.abspath('..'))

# We will reuse the basic loading function but handle the rest here for transparency
from src.data_processing import load_and_pivot_data

/Users/yaisanadone/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Config
RAW_DATA_PATH = '../data/prices_all.csv'
PROCESSED_DATA_PATH = '../data/prices_wide_clean.parquet'
START_DATE = '2021-10-01'
REGIONS = ['NSW1', 'QLD1', 'SA1', 'TAS1', 'VIC1']

## 1. Load & Pivot

Using our utility function to get the initial wide format.

In [3]:
df_wide = load_and_pivot_data(RAW_DATA_PATH, start_date=START_DATE)
df_wide = df_wide.set_index('date_time')  # Set index for time series manipulation

print(f"Initial shape: {df_wide.shape}")
df_wide.head()

Loading data from ../data/prices_all.csv...
Data loaded. Shape: (348372, 6)
Initial shape: (348372, 5)


,NSW1,QLD1,SA1,TAS1,VIC1
date_time,,,,,
2021-10-01 00:00:00,48.10,47.07,23.52,21.28,24.83
2021-10-01 00:05:00,43.64,43.09,17.22,16.23,17.99
2021-10-01 00:10:00,54.74,54.00,16.97,16.23,17.99
2021-10-01 00:15:00,62.43,61.15,8.57,8.07,8.95
2021-10-01 00:20:00,55.01,54.56,8.40,8.07,8.95


## 2. Data Quality: Time Continuity Check

In time-series forecasting, missing timestamps can break lag calculations. We must ensure a continuous frequency.

In [4]:
# Infer frequency (usually 5T for 5-min or 30T for 30-min NEM data)
# Note: Recent NEM data is 5-min settlement. Older is 30-min.
inferred_freq = pd.infer_freq(df_wide.index[:100])
print(f"Inferred frequency: {inferred_freq}")

if inferred_freq is None:
    # Fallback/Force 5 min if not detectable (usually due to gaps)
    inferred_freq = '5min'
    print(f"Frequency could not be inferred. Assuming {inferred_freq}")

# Create a full range index
full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.index.max(), freq=inferred_freq)
print(f"Expected length: {len(full_idx)}")
print(f"Actual length:   {len(df_wide)}")

# Reindex check
if len(full_idx) != len(df_wide):
    print(f"WARNING: Missing {len(full_idx) - len(df_wide)} timestamps found. Reindexing...")
    df_wide = df_wide.reindex(full_idx)
else:
    print("Time index is complete.")

Inferred frequency: 5min
Expected length: 348383
Actual length:   348372


## 3. Handle Missing Values (Imputation)

Electricity prices rarely disappear; missing values usually mean a data collection issue. Forward-fill (`ffill`) is the standard safe assumption (price persists until updated).

In [5]:
nan_count = df_wide.isna().sum()
print("Missing values per region BEFORE imputation:")
print(nan_count)

if nan_count.sum() > 0:
    print("\nImputing with forward fill...")
    df_wide = df_wide.ffill()
    
    # Check remaining (if start of dataset was NaN)
    if df_wide.isna().sum().sum() > 0:
        print("Imputing remaining leading NaNs with backward fill...")
        df_wide = df_wide.bfill()

print("\nMissing values AFTER imputation:")
print(df_wide.isna().sum())

Missing values per region BEFORE imputation:
NSW1    11
QLD1    11
SA1     11
TAS1    11
VIC1    11
dtype: int64

Imputing with forward fill...

Missing values AFTER imputation:
NSW1    0
QLD1    0
SA1     0
TAS1    0
VIC1    0
dtype: int64


## 4. Save Processed Data

We save as Parquet for performance. If Parquet fails (missing dependencies), we fall back to CSV.

In [6]:
try:
    df_wide.to_parquet(PROCESSED_DATA_PATH)
    print(f"SUCCESS: Cleaned data saved to {PROCESSED_DATA_PATH}")
except ImportError:
    csv_path = PROCESSED_DATA_PATH.replace('.parquet', '.csv')
    df_wide.to_csv(csv_path)
    print(f"WARNING: PyArrow/FastParquet not found. Saved as CSV to {csv_path}")

SUCCESS: Cleaned data saved to ../data/prices_wide_clean.parquet
